In [ ]:
import os
import glob
import time

import pandas as pd

from dotenv import load_dotenv
from google import genai

0.設定

In [ ]:
load_dotenv()
gemini_api_key = os.getenv("API_KEY")

1.gemini apiの設定

In [ ]:
generation_config = {
    "temperature": 0.2,
    "max_output_tokens": 1500, 
}

model = "gemini-3-flash-preview" 

In [ ]:
client = genai.Client(api_key = gemini_api_key)

In [ ]:
def categorize(prompt):
    contents = prompt

    response = client.models.generate_content(
        model = model, 
        config = generation_config,
        contents = contents
    )

    # print(response.text)

    return response.text

2.ファイル読み込み、ラベリングデータ作成

In [ ]:
# 重複してラベリングしないように、
# ラベリング済みのコメントを保存したファイルのリストを作成
list_files = glob.glob("../data/gemini_response/*.csv")

In [ ]:
list_files.sort()
list_files

In [ ]:
# 保存済みのレスポンスを読み込み
df_done  = pd.DataFrame()

for file_path in list_files:
    df = pd.read_csv(file_path)
    df_done  = pd.concat([df_done , df])
    df_done .reset_index(drop = True, inplace = True)

In [ ]:
df_done

In [ ]:
# ラベリング前のyahooコメントを読み込み
df_raw = pd.read_csv("../data/yahoo_comments_renumbering.csv")

In [ ]:
# 未処理のidを抽出
if not df_done.empty:
    list_notyet_ids = list(set(df_raw["id"].to_list()) - set(df_done["id"].to_list()))
else:
    list_notyet_ids = df_raw["id"].to_list()

In [ ]:
# 未処理（末ラベリング）のコメントを取り出す。
df_filterd = df_raw[df_raw["id"].isin(list_notyet_ids)]

In [ ]:
# 未処理（末ラベリング）が抽出できているか確認
df_filterd

In [ ]:
# バッチサイズを設定
# バッチの始まりとなるindexを取得
batch_size = 5
list_batch_indexes = [i for i in range(0, len(df_filterd), batch_size)]

In [ ]:
# バッチの最初のindexがバッチサイズ間隔になっているか確認
list_batch_indexes

3.プロンプトをgeminiになげる

In [ ]:
def make_prompt(comments):
    comments_csv = comments
    
    prompt = prompt = f"""
        あなたはSNSコメント分析の専門家です。

        以下のコメントをそれぞれ独立に読み、行政・税金に関する不満の種類を1つだけ分類してください。

        【重要ルール】
        - 各コメントは完全に独立して判断すること
        - 他のコメントの影響を受けないこと
        - 必ず指定フォーマットのみを出力すること
        - 説明・補足・前置きは禁止
        - 出力はCSV形式のみ

        【分類カテゴリ】
        1. 無駄遣い：税金の使い道・支出内容が無駄だと批判
        2. 不透明性：説明不足・意思決定の不明確さへの不満
        3. 不公平：負担や恩恵の偏りへの不満
        4. 強い批判：怒り・攻撃的表現が主軸（感情が中心の場合のみ）
        5. その他：中立・事実・軽い意見

        【出力ルール（最重要）】
        以下の形式のみを出力すること：

        <id>,<label>,<confidence>

        【出力例】
        1,無駄遣い,0.92
        2,不透明性,0.88
        3,不公平,0.95

        【厳格ルール】
        - ヘッダー禁止
        - JSON禁止
        - Markdown禁止
        - 余計な文章禁止
        - 空行禁止
        - 各行は必ず3要素
        - confidenceは0〜1の小数（厳密でなくてよい）

        【入力データ】
        {comments_csv}
        """

    return prompt
    

In [ ]:
# df = df_yahoo_comments[0:5]
# comments_csv = df[["id","comment_text"]].to_csv(index=False)

In [ ]:
# ループでバッチごとにapiをたたく

for i in list_batch_indexes:

    # list_label = []
    if i == list_batch_indexes[-1]:
        pass
    else:
        df = df_filterd[i:i+batch_size]
        comments_csv = df[["id","comment_text"]].to_csv(index=False)

        prompt = make_prompt(comments_csv)

        # labeling = categorize(str(prompt))
        labeling = categorize(prompt)

        lines = labeling.split("\n")
        # lines = [line for line in labeling.split("\n") if line.strip() and line != "None"]

        #出力の行がバッチサイズと一致しない時に再試行
        if len(lines) != batch_size:
            labeling = categorize(str(prompt))
            lines = labeling.split("\n")

        # list_label.extend(lines)

        l = []
        for row in lines: #バッチごとに分解する時
        # for row in list_label: 複数バッチの分解の時
            l.append(row.split(","))

        # 途中でエラーが起きた場合に備え、それまでの結果をcsvで保存
        # バッチごとにスタートのid起点でファイル名を分ける形で保存。
        # 変な挙動をしたバッチがあればあとでそこだけを排除できる
        file_path = f"../data/gemini_response/start_id_{df["id"].iloc[0]}.csv"
        pd.DataFrame(l, columns=["id","label","confidence"]).to_csv(file_path, index=False)

        # 動作確認で一回だけ回す
        break
        
        # apiの１分回の利用制限にひっかからないようにsleep
        time.sleep(60)

# print(labeling)
# print(type(labeling))
# print(labeling.text if hasattr(labeling, "text") else "no text")

# print(comments_csv)
# print(labeling)
# print(lines)
        

In [ ]:
# レスポンスの中身の確認
# list_label[0].split(",")

In [ ]:
# l = []
# for str in list_label:
#     l.append(str.split(","))

# pd.DataFrame(l, columns=["id","label","confidence"]).to_csv("../data/gemini_responce.csv", index=False)

In [ ]:
# forループでバッチごとに回す前に単発で動作確認
# comments_csv='''
# id,text
# 1,"Why are billions of tax dollars being poured into projects that never even get completed? This feels like blatant waste."
# 2,"The government released a new budget but didn’t explain half of the spending categories. Where is the transparency?"
# '''

In [ ]:
# # forループでバッチごとに回す前に単発で動作確認

# comments_csv='''
# id,text
# 1,"Why are billions of tax dollars being poured into projects that never even get completed? This feels like blatant waste."
# 2,"The government released a new budget but didn’t explain half of the spending categories. Where is the transparency?"
# '''

# prompt = prompt = f"""
# あなたはSNSコメント分析の専門家です。

# 以下のコメントをそれぞれ独立に読み、行政・税金に関する不満の種類を1つだけ分類してください。

# 【重要ルール】
# - 各コメントは完全に独立して判断すること
# - 他のコメントの影響を受けないこと
# - 必ず指定フォーマットのみを出力すること
# - 説明・補足・前置きは禁止
# - 出力はCSV形式のみ

# 【分類カテゴリ】
# 1. 無駄遣い：税金の使い道・支出内容が無駄だと批判
# 2. 不透明性：説明不足・意思決定の不明確さへの不満
# 3. 不公平：負担や恩恵の偏りへの不満
# 4. 強い批判：怒り・攻撃的表現が主軸（感情が中心の場合のみ）
# 5. その他：中立・事実・軽い意見

# 【出力ルール（最重要）】
# 以下の形式のみを出力すること：

# <id>,<label>,<confidence>

# 【出力例】
# 1,無駄遣い,0.92
# 2,不透明性,0.88
# 3,不公平,0.95

# 【厳格ルール】
# - ヘッダー禁止
# - JSON禁止
# - Markdown禁止
# - 余計な文章禁止
# - 空行禁止
# - 各行は必ず3要素
# - confidenceは0〜1の小数（厳密でなくてよい）

# 【入力データ】
# {comments_csv}
# """

In [ ]:
# categorize(str(prompt))